# Task 4: Game Title Detection (CPU fallback)

- **Type:** 5-class image classification, 230×120 JPEG crops. **Metric:** macro F1. Pretrained models allowed.
- **This notebook = CPU fallback:** frozen ImageNet ConvNeXt-Tiny embeddings (`src/features.py`, cached)
  + logistic regression, 5-fold StratifiedGroupKFold (near-duplicate groups by average hash).
  The GPU fine-tuning version is `task4_kaggle.ipynb`.
- **Labels:** train.csv holds strings; submitted integers follow the datacard
  `{7-11k:0, fiveN:1, freefried:2, pube:3, roblock:4}` (= alphabetical; see README "Conflicts").

Source: `src/task4_nb.py` (jupytext percent format). Experiments are logged in `experiments.md`.

In [1]:
import subprocess
import sys
import time
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "common").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, f1_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from common import metrics, paths, utils

TASK = 4
BACKBONE = "convnext_tiny"
LABELS = ["7-11k", "fiveN", "freefried", "pube", "roblock"]  # index = submitted integer (datacard order)
utils.seed_everything()
DATA = paths.task_data(TASK)
OUT = paths.outputs_dir(TASK)

## 1. Data card & load

In [2]:
print(paths.datacard(TASK)[:420])
train = pd.read_csv(DATA / "train.csv")
test = pd.read_csv(DATA / "test.csv")
assert set(train["class"]) == set(LABELS)
y = train["class"].map({c: i for i, c in enumerate(LABELS)}).to_numpy()
print("train", train.shape, "test", test.shape)
print(train["class"].value_counts(normalize=True).round(4))

TASK 4: GAME TITLE DETECTION

TARGET VARIABLE: class (Categorical)
Game title classification label - 5 different games
   0: 7-11 Knights
   1: FiveN
   2: Free Fried
   3: PUBE
   4: Roblock

IMAGE DATA:

1. id (object)
   Unique identifier linking to image files
   File mapping: train.csv → tasks/04_image_classification/data/train/{id}
train (31546, 2) test (25889, 2)
class
7-11k        0.3940
pube         0.1879
freefried    0.1774
fiveN        0.1397
roblock      0.1011
Name: proportion, dtype: float64


## 2. Embeddings + image metadata (cached; computed by `src/features.py` if missing, ~15 min on CPU)

In [3]:
f_tr, f_te = OUT / f"feat_{BACKBONE}_train.npy", OUT / f"feat_{BACKBONE}_test.npy"
if not (f_tr.exists() and f_te.exists()):
    subprocess.run([sys.executable, str(ROOT / "task4/src/features.py"), "--backbone", BACKBONE], check=True)
X, X_test = np.load(f_tr), np.load(f_te)
meta_tr, meta_te = pd.read_csv(OUT / "img_meta_train.csv"), pd.read_csv(OUT / "img_meta_test.csv")
assert (meta_tr["id"].astype(str).values == train["id"].astype(str).values).all()
assert (meta_te["id"].astype(str).values == test["id"].astype(str).values).all()
print(X.shape, X_test.shape)

(31546, 768) (25889, 768)


## 3. EDA (details in `experiments.md`)

In [4]:
print("image sizes train:", meta_tr.groupby(["width", "height"]).size().to_dict(),
      "| test:", meta_te.groupby(["width", "height"]).size().to_dict())
meta_tr["label"] = train["class"].values
print(meta_tr.groupby("label")[["mean_r", "mean_g", "mean_b"]].mean().round(3))
dup = meta_tr["ahash"].duplicated(keep=False)
mixed = (meta_tr[dup].groupby("ahash")["label"].nunique() > 1).sum()
cross = meta_te["ahash"].isin(set(meta_tr["ahash"])).mean()
print(f"train rows in a duplicate-hash group: {dup.mean():.3%} ({meta_tr['ahash'].nunique():,} unique hashes); "
      f"mixed-label groups: {mixed}; test rows whose hash appears in train: {cross:.3%}")

image sizes train: {(230, 120): 31546} | test: {(230, 120): 25889}
           mean_r  mean_g  mean_b
label                            
7-11k       0.476   0.478   0.477
fiveN       0.414   0.413   0.413
freefried   0.434   0.436   0.433
pube        0.427   0.427   0.430
roblock     0.471   0.463   0.468


train rows in a duplicate-hash group: 0.682% (31,411 unique hashes); mixed-label groups: 9; test rows whose hash appears in train: 0.050%


## 4. Cross-validation: logistic regression on standardised embeddings
Groups = identical average hash → near-duplicates stay in the same fold. Scaler fit inside each fold.

In [5]:
groups = meta_tr["ahash"].factorize()[0]
FOLDS = list(StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=utils.SEED).split(X, y, groups))


def run_cv(name, **lr_kw):
    t0 = time.time()
    oof, pred = np.zeros((len(X), 5)), np.zeros((len(X_test), 5))
    for tr, va in FOLDS:
        model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, random_state=utils.SEED, **lr_kw))
        model.fit(X[tr], y[tr])
        oof[va] = model.predict_proba(X[va])
        pred += model.predict_proba(X_test) / len(FOLDS)
    folds = [metrics.macro_f1(y[va], oof[va].argmax(1)) for _, va in FOLDS]
    print(f"{name:14s} macro F1 {metrics.macro_f1(y, oof.argmax(1)):.4f} +- {np.std(folds):.4f}  ({time.time() - t0:.0f}s)")
    utils.save_preds(TASK, name, oof, pred)
    return dict(name=name, score=metrics.macro_f1(y, oof.argmax(1)), std=np.std(folds), oof=oof, pred=pred)


runs = [run_cv("lr_C0.1", C=0.1), run_cv("lr_C0.1_bal", C=0.1, class_weight="balanced")]

lr_C0.1        macro F1 0.8413 +- 0.0021  (151s)


lr_C0.1_bal    macro F1 0.8332 +- 0.0043  (139s)


In [6]:
best = max(runs, key=lambda r: r["score"])
print("best:", best["name"])
pred_oof = best["oof"].argmax(1)
print(pd.DataFrame(confusion_matrix(y, pred_oof), index=LABELS, columns=LABELS))
print("per-class F1:", dict(zip(LABELS, f1_score(y, pred_oof, average=None).round(4))))
pd.DataFrame([{k: v for k, v in r.items() if k in ("name", "score", "std")} for r in runs]).to_csv(OUT / "cv_results.csv", index=False)

best: lr_C0.1
           7-11k  fiveN  freefried  pube  roblock
7-11k      11523    205        284   293      124
fiveN        273   3684        124   184      142
freefried    403    119       4567   444       63
pube         362    200        466  4743      155
roblock      202    158         90   165     2573
per-class F1: {'7-11k': np.float64(0.9148), 'fiveN': np.float64(0.8398), 'freefried': np.float64(0.8209), 'pube': np.float64(0.807), 'roblock': np.float64(0.824)}


## 5. Write `outputs/final_pred.csv` (integers 0–4, datacard order)

In [7]:
test_label = best["pred"].argmax(1)
print("test pred dist:", dict(zip(LABELS, np.bincount(test_label, minlength=5) / len(test_label))))
final = utils.write_final_pred(TASK, test["id"], test_label)
final.head()

test pred dist: {'7-11k': np.float64(0.2582564023330372), 'fiveN': np.float64(0.17903356637954343), 'freefried': np.float64(0.14979334852640117), 'pube': np.float64(0.27231642782649), 'roblock': np.float64(0.14060025493452818)}


,id,prediction
0,scyd5lz,0
1,slbh6z4,0
2,s3u8t8d,3
3,s0jrf0c,0
4,susbkwa,3
